# Ejercicio: Imputer y Scaler - ¿Por qué importa el orden? (SOLUCIÓN)

## Objetivo
En este ejercicio aprenderás la importancia de **escalar los datos antes de aplicar un imputer** cuando trabajas con variables de diferentes magnitudes.

## Contexto
Tienes datos de sensores industriales con valores faltantes (NaN). Las variables tienen escalas muy diferentes:
- **Temperatura**: 20-25°C
- **Presión**: ~101,000 Pascal
- **Humedad**: 45-50%
- **Vibración**: 0.4-0.7 Hz
- **Consumo energía**: 2-3 kWh

⚠️ **IMPORTANTE**: Hay algunos valores atípicos (outliers) en los datos que pueden afectar significativamente los resultados.

## Enfoques a comparar
1. **SimpleImputer (mean)** sin escalar
2. **SimpleImputer (mean)** con StandardScaler
3. **KNNImputer** sin escalar
4. **KNNImputer** con StandardScaler

## Paso 1: Importar librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.precision', 4)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Paso 2: Cargar y explorar los datos

In [ ]:
# Cargar los datos
df = pd.read_csv('datos_sensores.csv')

# Explorar las primeras filas
print("Primeras 10 filas del dataset:")
df.head(10)

In [ ]:
# Información general del dataset
print("Información del dataset:")
df.info()

In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas:")
df.describe()

In [ ]:
# Contar valores faltantes por columna
print("Valores faltantes por columna:")
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Columna': missing_values.index,
    'Valores faltantes': missing_values.values,
    'Porcentaje': missing_percent.values
})
missing_df[missing_df['Valores faltantes'] > 0]

## Paso 3: Identificar y mostrar registros con valores faltantes

**Estos son los registros ANTES de aplicar cualquier imputación:**

In [ ]:
# Separar las columnas numéricas (excluir sensor_id)
numeric_cols = df.columns[1:].tolist()

# Identificar registros con valores faltantes
registros_con_nan = df[df.isnull().any(axis=1)].copy()

print("="*120)
print("REGISTROS CON VALORES FALTANTES (ANTES DE IMPUTAR)")
print("="*120)
print(f"\nTotal de registros con valores faltantes: {len(registros_con_nan)} de {len(df)}")
print("\n")
registros_con_nan

In [ ]:
# Guardar los índices de los registros con NaN para comparación posterior
indices_con_nan = registros_con_nan.index.tolist()

print(f"Índices de registros con NaN: {indices_con_nan}")

## Paso 4: Visualizar la distribución de las variables

In [ ]:
# Crear boxplots para cada variable
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna(), vert=True)
    axes[i].set_title(f'{col}', fontsize=10)
    axes[i].set_ylabel('Valor')
    axes[i].grid(True, alpha=0.3)

if len(numeric_cols) < 6:
    axes[-1].set_visible(False)

plt.tight_layout()
plt.suptitle('Distribución de variables (observa las diferentes escalas)', y=1.02, fontsize=14)
plt.show()

print("\n⚠️ OBSERVACIÓN: Las variables tienen escalas muy diferentes!")
print("- Presión: ~101,000")
print("- Temperatura: ~22")
print("- Vibración: ~0.5")

## Paso 5: Enfoque 1 - SimpleImputer SIN escalar

In [ ]:
df_simple_sin_escalar = df.copy()
X_simple_sin_escalar = df_simple_sin_escalar[numeric_cols]

imputer_simple_sin_escalar = SimpleImputer(strategy='mean')
X_imputado_simple_sin_escalar = imputer_simple_sin_escalar.fit_transform(X_simple_sin_escalar)

df_imputado_simple_sin_escalar = pd.DataFrame(X_imputado_simple_sin_escalar, columns=numeric_cols)
df_imputado_simple_sin_escalar.insert(0, 'sensor_id', df['sensor_id'])

print("✅ SimpleImputer SIN escalar aplicado")
print("\nMedias calculadas (afectadas por outliers):")
print(pd.Series(imputer_simple_sin_escalar.statistics_, index=numeric_cols))

In [ ]:
print("="*120)
print("REGISTROS QUE TENÍAN NaN - DESPUÉS DE SimpleImputer SIN escalar")
print("="*120)
print("\n")
df_imputado_simple_sin_escalar.loc[indices_con_nan]

## Paso 6: Enfoque 2 - SimpleImputer CON StandardScaler

In [ ]:
df_simple_con_escalar = df.copy()
X_simple_con_escalar = df_simple_con_escalar[numeric_cols].values

scaler_simple = StandardScaler()
scaler_simple.fit(df_simple_con_escalar[numeric_cols].dropna())

X_escalado_simple = X_simple_con_escalar.copy()
for i in range(X_simple_con_escalar.shape[1]):
    col_data = X_simple_con_escalar[:, i]
    mask = ~np.isnan(col_data)
    X_escalado_simple[mask, i] = (col_data[mask] - scaler_simple.mean_[i]) / scaler_simple.scale_[i]

imputer_simple_con_escalar = SimpleImputer(strategy='mean')
X_imputado_escalado_simple = imputer_simple_con_escalar.fit_transform(X_escalado_simple)

X_imputado_simple_con_escalar = scaler_simple.inverse_transform(X_imputado_escalado_simple)

df_imputado_simple_con_escalar = pd.DataFrame(X_imputado_simple_con_escalar, columns=numeric_cols)
df_imputado_simple_con_escalar.insert(0, 'sensor_id', df['sensor_id'])

print("✅ SimpleImputer CON StandardScaler aplicado")
print("\nMedias de los datos escalados:")
print(pd.Series(imputer_simple_con_escalar.statistics_, index=numeric_cols))

In [ ]:
print("="*120)
print("REGISTROS QUE TENÍAN NaN - DESPUÉS DE SimpleImputer CON escalar")
print("="*120)
print("\n")
df_imputado_simple_con_escalar.loc[indices_con_nan]

## Paso 7: Enfoque 3 - KNNImputer SIN escalar

In [ ]:
df_knn_sin_escalar = df.copy()
X_knn_sin_escalar = df_knn_sin_escalar[numeric_cols]

imputer_knn_sin_escalar = KNNImputer(n_neighbors=5)
X_imputado_knn_sin_escalar = imputer_knn_sin_escalar.fit_transform(X_knn_sin_escalar)

df_imputado_knn_sin_escalar = pd.DataFrame(X_imputado_knn_sin_escalar, columns=numeric_cols)
df_imputado_knn_sin_escalar.insert(0, 'sensor_id', df['sensor_id'])

print("✅ KNNImputer SIN escalar aplicado (k=5 vecinos)")
print("\n⚠️ IMPORTANTE: KNN usa distancia euclidiana, las variables con mayor escala dominan.")

In [ ]:
print("="*120)
print("REGISTROS QUE TENÍAN NaN - DESPUÉS DE KNNImputer SIN escalar")
print("="*120)
print("\n")
df_imputado_knn_sin_escalar.loc[indices_con_nan]

## Paso 8: Enfoque 4 - KNNImputer CON StandardScaler

In [ ]:
df_knn_con_escalar = df.copy()
X_knn_con_escalar = df_knn_con_escalar[numeric_cols].values

scaler_knn = StandardScaler()
scaler_knn.fit(df_knn_con_escalar[numeric_cols].dropna())

X_escalado_knn = X_knn_con_escalar.copy()
for i in range(X_knn_con_escalar.shape[1]):
    col_data = X_knn_con_escalar[:, i]
    mask = ~np.isnan(col_data)
    X_escalado_knn[mask, i] = (col_data[mask] - scaler_knn.mean_[i]) / scaler_knn.scale_[i]

imputer_knn_con_escalar = KNNImputer(n_neighbors=5)
X_imputado_escalado_knn = imputer_knn_con_escalar.fit_transform(X_escalado_knn)

X_imputado_knn_con_escalar = scaler_knn.inverse_transform(X_imputado_escalado_knn)

df_imputado_knn_con_escalar = pd.DataFrame(X_imputado_knn_con_escalar, columns=numeric_cols)
df_imputado_knn_con_escalar.insert(0, 'sensor_id', df['sensor_id'])

print("✅ KNNImputer CON StandardScaler aplicado")
print("\n✨ Todas las variables tienen el mismo peso en el cálculo de distancias.")

In [ ]:
print("="*120)
print("REGISTROS QUE TENÍAN NaN - DESPUÉS DE KNNImputer CON escalar")
print("="*120)
print("\n")
df_imputado_knn_con_escalar.loc[indices_con_nan]

## Paso 9: Comparación detallada ANTES vs DESPUÉS (4 métodos)

In [ ]:
print("="*120)
print("COMPARACIÓN DETALLADA: ANTES vs DESPUÉS DE IMPUTACIÓN")
print("="*120)

for idx in indices_con_nan[:10]:
    print(f"\n{'='*120}")
    print(f"Sensor ID: {df.loc[idx, 'sensor_id']}")
    print(f"{'='*120}")
    
    for col in numeric_cols:
        original_val = df.loc[idx, col]
        
        if pd.isna(original_val):
            simple_sin = df_imputado_simple_sin_escalar.loc[idx, col]
            simple_con = df_imputado_simple_con_escalar.loc[idx, col]
            knn_sin = df_imputado_knn_sin_escalar.loc[idx, col]
            knn_con = df_imputado_knn_con_escalar.loc[idx, col]
            
            print(f"\n📊 {col}:")
            print(f"  {'ANTES (Original):':<35} NaN")
            print(f"  {'─'*80}")
            print(f"  {'DESPUÉS - SimpleImputer SIN escalar:':<35} {simple_sin:>12.4f}")
            print(f"  {'DESPUÉS - SimpleImputer CON escalar:':<35} {simple_con:>12.4f}")
            print(f"  {'DESPUÉS - KNNImputer SIN escalar:':<35} {knn_sin:>12.4f}")
            print(f"  {'DESPUÉS - KNNImputer CON escalar:':<35} {knn_con:>12.4f}")
            print(f"  {'─'*80}")
            rango = abs(max(simple_sin, simple_con, knn_sin, knn_con) - min(simple_sin, simple_con, knn_sin, knn_con))
            print(f"  {'Rango de diferencia entre métodos:':<35} {rango:.4f}")

## Paso 10: Tabla comparativa consolidada

In [ ]:
print("="*120)
print("TABLA COMPARATIVA: VALORES IMPUTADOS POR VARIABLE")
print("="*120)

for col in numeric_cols:
    indices_nan_col = df[df[col].isna()].index
    
    if len(indices_nan_col) > 0:
        print(f"\n{'='*120}")
        print(f"Variable: {col}")
        print(f"{'='*120}")
        
        comparacion = pd.DataFrame({
            'Sensor ID': df.loc[indices_nan_col, 'sensor_id'].values,
            'ANTES': ['NaN'] * len(indices_nan_col),
            'Simple SIN escalar': df_imputado_simple_sin_escalar.loc[indices_nan_col, col].values,
            'Simple CON escalar': df_imputado_simple_con_escalar.loc[indices_nan_col, col].values,
            'KNN SIN escalar': df_imputado_knn_sin_escalar.loc[indices_nan_col, col].values,
            'KNN CON escalar': df_imputado_knn_con_escalar.loc[indices_nan_col, col].values
        })
        
        print(comparacion.to_string(index=False))

## Paso 11: Análisis estadístico de las diferencias

In [ ]:
print("\n" + "="*120)
print("RESUMEN: MEDIAS CALCULADAS POR CADA VARIABLE")
print("="*120)

resumen = pd.DataFrame({
    'Original (con NaN)': df[numeric_cols].mean(),
    'SimpleImputer sin escalar': df_imputado_simple_sin_escalar[numeric_cols].mean(),
    'SimpleImputer con escalar': df_imputado_simple_con_escalar[numeric_cols].mean(),
    'KNNImputer sin escalar': df_imputado_knn_sin_escalar[numeric_cols].mean(),
    'KNNImputer con escalar': df_imputado_knn_con_escalar[numeric_cols].mean()
})

print(resumen)

print("\n" + "="*120)
print("DESVIACIONES ESTÁNDAR")
print("="*120)

resumen_std = pd.DataFrame({
    'Original (con NaN)': df[numeric_cols].std(),
    'SimpleImputer sin escalar': df_imputado_simple_sin_escalar[numeric_cols].std(),
    'SimpleImputer con escalar': df_imputado_simple_con_escalar[numeric_cols].std(),
    'KNNImputer sin escalar': df_imputado_knn_sin_escalar[numeric_cols].std(),
    'KNNImputer con escalar': df_imputado_knn_con_escalar[numeric_cols].std()
})

print(resumen_std)

## Paso 12: Visualización comparativa

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    
    mask_not_nan = ~df[col].isna()
    ax.scatter(df[mask_not_nan].index, df.loc[mask_not_nan, col], 
               alpha=0.6, label='Original', s=50, color='gray')
    
    mask_nan = df[col].isna()
    
    if mask_nan.sum() > 0:
        ax.scatter(df[mask_nan].index, df_imputado_simple_sin_escalar.loc[mask_nan, col], 
                   alpha=0.8, label='Simple sin escalar', marker='o', s=100, color='red')
        
        ax.scatter(df[mask_nan].index, df_imputado_simple_con_escalar.loc[mask_nan, col], 
                   alpha=0.8, label='Simple con escalar', marker='s', s=100, color='blue')
        
        ax.scatter(df[mask_nan].index, df_imputado_knn_sin_escalar.loc[mask_nan, col], 
                   alpha=0.8, label='KNN sin escalar', marker='^', s=100, color='orange')
        
        ax.scatter(df[mask_nan].index, df_imputado_knn_con_escalar.loc[mask_nan, col], 
                   alpha=0.8, label='KNN con escalar', marker='D', s=100, color='green')
    
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel('Índice del sensor')
    ax.set_ylabel('Valor')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Comparación de métodos de imputación', y=1.01, fontsize=16, fontweight='bold')
plt.show()

## Paso 13: Detectar y analizar outliers

In [ ]:
def detectar_outliers_iqr(data, columna):
    Q1 = data[columna].quantile(0.25)
    Q3 = data[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = data[(data[columna] < limite_inferior) | (data[columna] > limite_superior)]
    return outliers, limite_inferior, limite_superior

print("="*120)
print("DETECCIÓN DE OUTLIERS (Método IQR)")
print("="*120)

for col in numeric_cols:
    outliers, lim_inf, lim_sup = detectar_outliers_iqr(df, col)
    if len(outliers) > 0:
        print(f"\n{col}:")
        print(f"  Límite inferior: {lim_inf:.2f}")
        print(f"  Límite superior: {lim_sup:.2f}")
        print(f"  Outliers detectados: {len(outliers)}")
        print(outliers[['sensor_id', col]])
        print(f"\n  ⚠️ Estos outliers afectan significativamente la media calculada!")

## Paso 14: Impacto de los outliers

In [ ]:
print("="*120)
print("IMPACTO DE LOS OUTLIERS EN LAS MEDIAS")
print("="*120)

for col in numeric_cols:
    media_con_outliers = df[col].mean()
    mediana = df[col].median()
    
    outliers, lim_inf, lim_sup = detectar_outliers_iqr(df, col)
    df_sin_outliers = df[(df[col] >= lim_inf) & (df[col] <= lim_sup)]
    media_sin_outliers = df_sin_outliers[col].mean()
    
    diferencia = abs(media_con_outliers - media_sin_outliers)
    porcentaje_diff = (diferencia / media_sin_outliers * 100) if media_sin_outliers != 0 else 0
    
    print(f"\n{col}:")
    print(f"  Media con outliers:    {media_con_outliers:>12.4f}")
    print(f"  Media sin outliers:    {media_sin_outliers:>12.4f}")
    print(f"  Mediana:               {mediana:>12.4f}")
    print(f"  Diferencia absoluta:   {diferencia:>12.4f}")
    print(f"  Diferencia porcentual: {porcentaje_diff:>12.2f}%")
    
    if porcentaje_diff > 10:
        print(f"  ⚠️ ¡IMPACTO SIGNIFICATIVO! Los outliers sesgan la media más del 10%")

## Conclusiones

### Hallazgos principales:

#### 1. **SimpleImputer vs KNNImputer**
- **SimpleImputer (mean)**: Usa la media global. Rápido pero ignora relaciones entre variables.
- **KNNImputer**: Usa k vecinos cercanos. Considera relaciones pero muy sensible a escalas.

#### 2. **Impacto del escalado**
- **Sin escalar**: Variables con mayor magnitud (presión ~101,000) dominan los cálculos de distancia en KNN.
- **Con escalar**: Todas las variables tienen el mismo peso.

#### 3. **Efecto de los outliers**
- Los outliers extremos sesgan significativamente:
  - La media calculada por SimpleImputer
  - Los vecinos seleccionados por KNNImputer sin escalar
- El escalado ayuda a mitigar parcialmente este efecto.

#### 4. **Diferencias en los resultados**
- **SimpleImputer sin escalar**: Valores imputados muy sesgados por outliers
- **SimpleImputer con escalar**: Valores más razonables pero aún afectados
- **KNNImputer sin escalar**: Dominado por la variable de mayor escala
- **KNNImputer con escalar**: Resultados más equilibrados y contextuales

### Recomendaciones:

1. **Siempre escala antes de usar KNNImputer** cuando las variables tienen diferentes magnitudes
2. **Detecta y maneja outliers** antes de imputar valores faltantes
3. **Considera usar mediana** en lugar de media si hay outliers significativos
4. **KNNImputer con escalado** suele dar mejores resultados cuando hay correlaciones entre variables
5. **Documenta tu pipeline** de preprocesamiento para reproducibilidad

### Pipeline recomendado:

```python
# 1. Detectar y manejar outliers
# 2. Escalar los datos (StandardScaler)
# 3. Aplicar KNNImputer
# 4. Revertir el escalado
# 5. Validar los resultados
```